In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-01-13 12:02:03--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-01-13 12:02:03 (147 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
f = open('input.txt', 'r', encoding='utf-8')
res = f.read()
print(f'type={type(res)}, len={len(res)}')

type=<class 'str'>, len=1115394


In [ ]:
# build vacab and encode/decode
vocab = sorted([k for k in set(res)])
vocab_reverse = {ch: idx for idx,ch in enumerate(vocab)}
def encode(s: str) -> list:
  return [vocab_reverse[ch] for ch in s]
def decode(l: list) -> str:
  return ''.join([vocab[idx] for idx in l])


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

batch_size = 32
vocab_size = len(vocab)
emb_size = 128
time_size = 128
device = 'cpu'
# 自动选择最优设备
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'current device is `{device}`')


current device is `cuda`


In [ ]:
import random

def get_train_data(input: str, evaluate_percentage=0.1):
  codes = encode(input)
  step_size = time_size//2
  data = []
  for i in range(0, len(codes) - time_size - 1, step_size):
    data.append(codes[i:i+time_size+1])
  random.shuffle(data)
  x = torch.tensor([d[:-1] for d in data]).to(device)
  y = torch.tensor([d[1:] for d in data]).to(device)
  eval_pivot = int(x.shape[0]*(1-evaluate_percentage))
  x_train, y_train = x[:eval_pivot], y[:eval_pivot]
  x_eval, y_eval = x[eval_pivot:], y[eval_pivot:]
  return x_train,y_train,x_eval,y_eval

  # make x and y

x_train,y_train,x_eval,y_eval = get_train_data(res)
print(f'train: {x_train.shape} and {y_train.shape}')
print(f'eval:  {x_eval.shape} and {y_eval.shape}')

train: torch.Size([15684, 128]) and torch.Size([15684, 128])
eval:  torch.Size([1743, 128]) and torch.Size([1743, 128])


In [ ]:

masked = torch.triu(torch.full((time_size,time_size), float('-inf')), diagonal=1).to(device)

class Transformer(nn.Module):
  def __init__(self, emb_size, head_count):
    super().__init__()
    self.emb_size = emb_size
    self.head_count = head_count
    self.Wq = nn.Linear(emb_size, emb_size, bias=False)
    self.Wk = nn.Linear(emb_size, emb_size, bias=False)
    self.Wv = nn.Linear(emb_size, emb_size, bias=False)
    self.Wo = nn.Linear(emb_size, emb_size, bias=False)
    self.ln1 = nn.LayerNorm(normalized_shape=emb_size)
    self.ln2 = nn.LayerNorm(normalized_shape=emb_size)
    self.Drop = nn.Dropout(0.1)
    self.FFN = nn.Sequential(
        nn.Linear(emb_size, 4*emb_size),
        nn.GELU(),
        nn.Dropout(0.1),
        nn.Linear(4*emb_size, emb_size),
        nn.Dropout(0.1)
    )

  def forward(self, x):
    # Part1: Attention
    residual = x
    x = self.ln1(x)
    B,T,C = x.shape
    head_size = C//self.head_count

    q = self.Wq(x)
    k = self.Wk(x)
    v = self.Wv(x) # (B,T,C)@(C,C) = (B,T,C)

    # 1. split to multi heads
    q = q.view(B, T, self.head_count, head_size).transpose(1,2)
    k = k.view(B, T, self.head_count, head_size).transpose(1,2)
    v = v.view(B, T, self.head_count, head_size).transpose(1,2)

    # 2. compute softmax
    weights = torch.matmul(q, k.transpose(-2,-1))/(head_size**0.5)
    weights = weights+masked[:T, :T]
    scores = F.softmax(weights, dim=-1)
    scores = self.Drop(scores)

    # 3. cancat and output
    output = torch.matmul(scores, v).transpose(1,2) # (B,H,T,T)@(B,H,T,C/H) = (B,H,T,C/h) = (B,T,C)
    #print(f'attentions_shape={output.shape}')
    output = output.reshape(B, T, C)
    #print(f'output_shape={output.shape}')
    output = self.Wo(output) # (B,T,C)
    output = self.Drop(output)
    output = output+residual

    # Part2: FFN
    residual = output
    output = self.ln2(output)
    output = self.FFN(output)
    output = output+residual

    return output


In [ ]:
# Model definition
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.vocab_size = vocab_size
    self.token_embedding_layer = nn.Embedding(vocab_size, emb_size)
    self.position_layer = nn.Embedding(time_size, emb_size) # position embedding
    self.transformor_layers = nn.Sequential(
        Transformer(emb_size, 8),
        Transformer(emb_size, 8),
        Transformer(emb_size, 8),
        Transformer(emb_size, 8),
    )
    self.norm_layer = nn.LayerNorm(normalized_shape=emb_size)
    self.output_layer = nn.Linear(emb_size, vocab_size)
    self.do = nn.Dropout(0.1)

  def forward(self, x):
    B,T = x.shape
    # x=[B,T]
    x = self.token_embedding_layer(x)
    # x=[B,T,C]
    try:
      positions = self.position_layer(torch.arange(T, device=x.device))
    except Exception as e:
      print(f'position_layer={time_size},{emb_size}. T={T}')
      raise e
    # positions=[T,C]
    x = x + positions
    x = self.do(x)
    x = self.transformor_layers(x)
    x = self.norm_layer(x)
    x = self.output_layer(x)
    # x=[B,T,vocab_size]
    return x

  def generate(self, s, max_new_tokens=80):
    idx = encode(s)
    idx = torch.tensor(idx).unsqueeze(0).to(device)
    self.eval()
    with torch.no_grad():
      for _ in range(max_new_tokens):
        if idx.shape[1] >= time_size:
          print(f'BREAK: require too much tokens')
          break
        o = self.forward(idx)
        o = F.softmax(o, dim=-1)
        o = o[:,-1,:]
        newidx = torch.multinomial(o, num_samples=1)
        idx = torch.cat((idx, newidx), dim=1)
      print(f'[final]{decode(idx[0].tolist())}')



NameError: name 'nn' is not defined

In [ ]:
from torch.optim import AdamW
import numpy as np
import time

m = Model()
m = m.to(device)
optimizer = AdamW(m.parameters(), lr=3e-4)
epoch_nums = 20
train_loss = 0
eval_loss = 0
train_steps = len(x_train)//batch_size
eval_steps =  len(x_eval)//batch_size

start = time.perf_counter()
#train
for epoch in range(epoch_nums):
  # ===== 每个epoch开始时重置 =====
  m.train()  # 切换到训练模式
  train_loss = 0  # 重置loss
  train_steps = 0  # 计数器
  for i in range(0, len(x_train)-batch_size, batch_size):
    input = x_train[i:i+batch_size]
    target = y_train[i:i+batch_size]
    predict = m(input)

    loss = F.cross_entropy(predict.permute(0,2,1), target)
    # 累积loss
    train_loss += loss.item()
    train_steps += 1

    # backward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    # 梯度裁剪
    torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
    optimizer.step()

  # Epoch结束，打印平均loss
  avg_train_loss = train_loss / train_steps
  print(f'[Epoch {epoch:2d}][Train Summary] Avg Loss: {avg_train_loss:.4f}')

  #eval
  m.eval()
  with torch.no_grad():
    eval_loss = 0
    for i in range(0, len(x_eval)-batch_size, batch_size):
      input = x_eval[i:i+batch_size]
      target = y_eval[i:i+batch_size]
      predict = m(input)
      loss = F.cross_entropy(predict.permute(0,2,1), target)
      eval_loss += loss.item()
    print(f'[Evaluate]loss={eval_loss/eval_steps: .4f}')
  end_time = time.perf_counter()
  print(f'[Epoch {epoch:2d}] time: {(end_time-start)/60: .1f}min')


[Epoch  0][Train Summary] Avg Loss: 2.6332
[Evaluate]loss= 2.4321
[Epoch  0] time:  0.2min
[Epoch  1][Train Summary] Avg Loss: 2.3782
[Evaluate]loss= 2.2387
[Epoch  1] time:  0.3min
[Epoch  2][Train Summary] Avg Loss: 2.2224
[Evaluate]loss= 2.0894
[Epoch  2] time:  0.5min
[Epoch  3][Train Summary] Avg Loss: 2.1049
[Evaluate]loss= 1.9789
[Epoch  3] time:  0.7min
[Epoch  4][Train Summary] Avg Loss: 2.0124
[Evaluate]loss= 1.8852
[Epoch  4] time:  0.8min
[Epoch  5][Train Summary] Avg Loss: 1.9360
[Evaluate]loss= 1.8120
[Epoch  5] time:  1.0min
[Epoch  6][Train Summary] Avg Loss: 1.8720
[Evaluate]loss= 1.7537
[Epoch  6] time:  1.2min
[Epoch  7][Train Summary] Avg Loss: 1.8192
[Evaluate]loss= 1.7049
[Epoch  7] time:  1.3min
[Epoch  8][Train Summary] Avg Loss: 1.7750
[Evaluate]loss= 1.6645
[Epoch  8] time:  1.5min
[Epoch  9][Train Summary] Avg Loss: 1.7390
[Evaluate]loss= 1.6311
[Epoch  9] time:  1.6min
[Epoch 10][Train Summary] Avg Loss: 1.7068
[Evaluate]loss= 1.6039
[Epoch 10] time:  1.8min

In [ ]:
m.generate('This is a beautiful girl ')

[final]This is a beautiful girl hath Rome, form this brother
A should thou ceorss of sceeconier; theat me to tha


In [ ]:
# Here is the loss history:
# MLP
loss=2.4815752506256104

# only one attention layer
loss=2.5528461933135986

# add MHA
loss=2.428689479827881

# add position
loss=0.91947537660598752

# add position & masked,
loss=2.407407283782959

# add residual & layer_norm
loss=2.395630359649658

# add FFN
loss=2.1918935775756836

# optimize codes and add more dropout
loss=2.227206230163574

# add 4*attention layers
loss=2.0844688415527344

# add more dropouts and 2*attention layers
loss=2.5779290199279785

# optimize train loop
loss= 2.4971
cpu = '8m'
gpu = '0.3m'

# change lr=1e-4 -> 3e-4, attention_layers=2->4, epoches=2->20
# batch=5->32, emb_size=80->128, add gradient_clip
loss = 1.54